In [11]:
import cv2
import numpy as np

from Food_detection import FoodDetection
from Spoon_detection import SpoonDetection
from FoodAreap import FoodSegmenter
from SpoonLength import SpoonDepthEstimator
from ultralytics import YOLO

class FoodAreaPipeline:
    """
    Pipeline:
    1. Detect food type.
    2. Detect spoon/fork.
    3. Segment the detected utensil.
    4. Get Depth Pro depth map.
    5. Calibrate utensil length to 15 cm and create a
       depth-adjusted cm/pixel map.
    6. Segment each food and estimate its visible area in cm².

    The depth-adjusted area is an approximation. It does not
    estimate food volume or mass.
    """

    UTENSIL_LENGTH_CM = 15.0
    CONE_HEIGHT_CM = 5.0

    def __init__(
        self,
        food_model_path,
        utensil_model_path,
        sam_model_path=r"C:\Study\Major\sam3.pt",
        depth_api_url="http://localhost:8000/predict-depth-map",
        food_confidence=0.50,
        utensil_confidence=0.25,
        ingredient_model_path=None,
    ):
        print("\nInitializing food area pipeline...")

        # Stage 1: food detector
        self.food_detector = FoodDetection(
            food_model_path
        )

        # Stage 2: spoon/fork detector
        self.utensil_detector = SpoonDetection(
            utensil_model_path
        )

        # FoodSegmenter loads YOLO + SAM 3.
        # Its SAM predictor will also be reused for utensil masks.
        self.food_segmenter = FoodSegmenter(
            yolo_model_path=food_model_path,
            sam_model_path=sam_model_path,
            conf=food_confidence,
        )

        # Reuse the existing Depth Pro FastAPI client.
        # This class also loads a SAM model internally, although
        # this pipeline only uses its API method.
        self.depth_estimator = SpoonDepthEstimator(
            sam_model_path=sam_model_path,
            depth_api_url=depth_api_url,
        )

        self.utensil_confidence = utensil_confidence

        print("Pipeline initialized.")
        # Ingredient/component detection model from masking.ipynb
        if ingredient_model_path is None:
            raise ValueError(
                "Pass ingredient_model_path for ingredient analysis."
            )

        self.ingredient_model = YOLO(ingredient_model_path)

    # ---------------------------------------------------------
    # UTILITY: NORMALIZE A SAM MASK TO ORIGINAL IMAGE SIZE
    # ---------------------------------------------------------
    @staticmethod
    def _resize_mask(mask, image_shape):
        height, width = image_shape[:2]

        if mask.shape != (height, width):
            mask = cv2.resize(
                mask.astype(np.uint8),
                (width, height),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)

        return mask.astype(bool)

    # ---------------------------------------------------------
    # UTILITY: SELECT THE MASK THAT BEST MATCHES A DETECTION
    # ---------------------------------------------------------
    @classmethod
    def _select_mask_for_box(
        cls,
        masks,
        bbox,
        image_shape,
    ):
        """
        Select the SAM mask with the greatest fraction of its
        pixels inside the detector's bounding box.
        """
        x1, y1, x2, y2 = map(int, bbox)

        height, width = image_shape[:2]

        x1 = max(0, min(x1, width - 1))
        x2 = max(x1 + 1, min(x2, width))
        y1 = max(0, min(y1, height - 1))
        y2 = max(y1 + 1, min(y2, height))

        best_mask = None
        best_score = -1.0

        for raw_mask in masks:
            mask = cls._resize_mask(
                raw_mask,
                image_shape,
            )

            mask_area = int(mask.sum())

            if mask_area == 0:
                continue

            inside_box = mask[y1:y2, x1:x2]
            overlap_pixels = int(inside_box.sum())

            # Prefer masks that substantially overlap the
            # corresponding detection.
            score = overlap_pixels / mask_area

            if score > best_score:
                best_score = score
                best_mask = mask

        if best_mask is None or best_score <= 0:
            raise RuntimeError(
                "Could not find a SAM mask overlapping "
                "the detection bounding box."
            )

        return best_mask

    # ---------------------------------------------------------
    # STAGE 1: DETECT FOOD TYPE
    # ---------------------------------------------------------
    def detect_food(self, image_path):
        print("\n[STAGE 1] Detecting food types...")

        foods = self.food_detector.detect_food(
            image_path
        )

        for i, food in enumerate(foods):
            print(
                f"  Food {i + 1}: {food['name']} "
                f"(confidence={food['confidence']:.3f})"
            )

        return foods

    # ---------------------------------------------------------
    # STAGE 2: DETECT SPOON OR FORK
    # ---------------------------------------------------------
    def detect_utensil(self, image_path):
        print("\n[STAGE 2] Detecting spoon/fork...")

        model = self.utensil_detector.model
        results = model(image_path)

        detections = []

        # Supports a YOLO model trained with classes named
        # "spoon" and/or "fork".
        for result in results:
            if result.boxes is None:
                continue

            for box in result.boxes:
                confidence = float(box.conf[0])

                if confidence < self.utensil_confidence:
                    continue

                class_id = int(box.cls[0])
                names = model.names

                if isinstance(names, dict):
                    name = str(names[class_id])
                else:
                    name = str(names[class_id])

                normalized_name = name.lower().strip()

                if (
                    "spoon" not in normalized_name
                    and "fork" not in normalized_name
                ):
                    continue

                x1, y1, x2, y2 = map(
                    float,
                    box.xyxy[0].tolist(),
                )

                detections.append({
                    "name": normalized_name,
                    "class_id": class_id,
                    "confidence": confidence,
                    "bbox": (x1, y1, x2, y2),
                })

        if not detections:
            raise RuntimeError(
                "No spoon/fork detected. Check that your "
                "utensil YOLO model has spoon/fork classes."
            )

        # Use the highest-confidence detected utensil.
        utensil = max(
            detections,
            key=lambda item: item["confidence"],
        )

        print(
            f"  Utensil: {utensil['name']} "
            f"(confidence={utensil['confidence']:.3f})"
        )

        return utensil

    # ---------------------------------------------------------
    # STAGE 3: SEGMENT THE DETECTED UTENSIL USING SAM 3
    # ---------------------------------------------------------
    def segment_utensil(self, image_path, utensil):
        print("\n[STAGE 3] Segmenting utensil with SAM 3...")

        image = cv2.imread(image_path)

        if image is None:
            raise FileNotFoundError(image_path)

        predictor = self.food_segmenter.sam_predictor

        predictor.set_image(image_path)

        # Prompt according to the YOLO-detected class.
        results = predictor(
            text=[utensil["name"]]
        )

        if not results or results[0].masks is None:
            raise RuntimeError(
                f"SAM 3 failed to segment {utensil['name']}."
            )

        raw_masks = (
            results[0].masks.data
            .detach()
            .cpu()
            .numpy()
        )

        mask = self._select_mask_for_box(
            masks=raw_masks,
            bbox=utensil["bbox"],
            image_shape=image.shape,
        )

        if int(mask.sum()) < 2:
            raise RuntimeError(
                "The utensil mask contains too few pixels."
            )

        # Find approximate long-axis endpoints using PCA.
        # Coordinates are stored as (x, y).
        ys, xs = np.where(mask)
        points = np.column_stack((xs, ys)).astype(
            np.float32
        )

        center = points.mean(axis=0)
        centered = points - center

        _, _, vh = np.linalg.svd(
            centered,
            full_matrices=False,
        )

        principal_axis = vh[0]
        projections = centered @ principal_axis

        point1 = points[np.argmin(projections)]
        point2 = points[np.argmax(projections)]

        utensil_length_px = float(
            np.linalg.norm(point2 - point1)
        )

        if utensil_length_px <= 0:
            raise RuntimeError(
                "Could not calculate utensil pixel length."
            )

        print(f"  Utensil mask pixels: {int(mask.sum())}")
        print(f"  Utensil length: {utensil_length_px:.2f} px")

        return {
            "mask": mask,
            "length_px": utensil_length_px,
            "endpoint1_xy": tuple(
                map(float, point1)
            ),
            "endpoint2_xy": tuple(
                map(float, point2)
            ),
        }

    # ---------------------------------------------------------
    # STAGE 4: GET DEPTH PRO DEPTH MAP
    # ---------------------------------------------------------
    def get_depth_map(self, image_path, image_shape):
        print("\n[STAGE 4] Requesting Depth Pro map...")

        # Existing SpoonDepthEstimator API returns the depth map
        # as a NumPy array.
        depth_map = self.depth_estimator.get_depth_from_api(
            image_path
        )

        height, width = image_shape[:2]

        if depth_map.ndim != 2:
            raise RuntimeError(
                f"Expected a 2D depth map; got {depth_map.shape}."
            )

        if depth_map.shape != (height, width):
            depth_map = cv2.resize(
                depth_map.astype(np.float32),
                (width, height),
                interpolation=cv2.INTER_LINEAR,
            )

        depth_map = depth_map.astype(np.float32)

        valid = np.isfinite(depth_map) & (depth_map > 0)

        if not valid.any():
            raise RuntimeError(
                "Depth Pro returned no valid positive depths."
            )

        print(f"  Depth map shape: {depth_map.shape}")
        print(
            f"  Valid depth range: "
            f"{depth_map[valid].min():.4f} - "
            f"{depth_map[valid].max():.4f}"
        )

        return depth_map

    # ---------------------------------------------------------
    # STAGE 5: CALIBRATE SCALE USING 15 CM UTENSIL
    # ---------------------------------------------------------
    def calculate_scale_map(
        self,
        depth_map,
        utensil_mask,
        utensil_length_px,
    ):
        print("\n[STAGE 5] Calculating depth-adjusted scale...")

        valid = np.isfinite(depth_map) & (depth_map > 0)

        utensil_depths = depth_map[
            utensil_mask & valid
        ]

        if utensil_depths.size == 0:
            raise RuntimeError(
                "No valid depth values inside utensil mask."
            )

        # Median is more robust than the minimum/maximum depth
        # to individual noisy pixels.
        reference_depth = float(
            np.median(utensil_depths)
        )

        # Calibrate the observed 2D utensil length to 15 cm
        # at its median depth.
        reference_cm_per_pixel = (
            self.UTENSIL_LENGTH_CM / utensil_length_px
        )

        # Approximate perspective adjustment:
        # pixel scale is proportional to depth.
        scale_map = np.full(
            depth_map.shape,
            np.nan,
            dtype=np.float32,
        )

        scale_map[valid] = (
            reference_cm_per_pixel
            * depth_map[valid]
            / reference_depth
        )

        print(
            f"  Assumed utensil length: "
            f"{self.UTENSIL_LENGTH_CM:.2f} cm"
        )
        print(
            f"  Reference depth: {reference_depth:.4f}"
        )
        print(
            f"  Reference scale: "
            f"{reference_cm_per_pixel:.6f} cm/pixel"
        )
        print(
            f"  Scale range: "
            f"{np.nanmin(scale_map):.6f} - "
            f"{np.nanmax(scale_map):.6f} cm/pixel"
        )

        return {
            "scale_map": scale_map,
            "reference_depth": reference_depth,
            "reference_cm_per_pixel": (
                reference_cm_per_pixel
            ),
        }

    # ---------------------------------------------------------
    # STAGE 6: SEGMENT EACH FOOD AND ESTIMATE ITS AREA
    # ---------------------------------------------------------
    def segment_food_item(
        self,
        image_path,
        food,
        image_shape,
    ):
        predictor = self.food_segmenter.sam_predictor

        # Use the detected food class as the SAM prompt.
        predictor.set_image(image_path)

        results = predictor(
            text=["food"]
        )

        if not results or results[0].masks is None:
            raise RuntimeError(
                f"SAM 3 could not segment food."
            )

        raw_masks = (
            results[0].masks.data
            .detach()
            .cpu()
            .numpy()
        )

        mask = self._select_mask_for_box(
            masks=raw_masks,
            bbox=(
                food["x1"],
                food["y1"],
                food["x2"],
                food["y2"],
            ),
            image_shape=image_shape,
        )

        return mask

    def calculate_food_area(
        self,
        food_mask,
        scale_map,
    ):
        # Each pixel represents approximately:
        # (cm/pixel)^2 square centimetres.
        valid_pixels = (
            food_mask
            & np.isfinite(scale_map)
            & (scale_map > 0)
        )

        pixel_count = int(valid_pixels.sum())

        if pixel_count == 0:
            raise RuntimeError(
                "No food-mask pixels have valid depth."
            )

        pixel_area_map = np.square(scale_map)

        area_cm2 = float(
            np.sum(pixel_area_map[valid_pixels])
        )

        return {
            "area_pixels": pixel_count,
            "area_cm2": area_cm2,
            "food_mask": food_mask,
        }

    # ---------------------------------------------------------
    # STAGE 7: BUILD THE TOTAL FOOD MASK FROM EXISTING OUTPUTS
    # ---------------------------------------------------------
    def build_total_food_mask(self, food_results, image_shape):
        """
        Combine the SAM3 masks already produced in Stage 6.
        Does not call SAM3.
        """
        height, width = image_shape[:2]
        total_food_mask = np.zeros(
            (height, width), dtype=bool
        )

        for food in food_results:
            mask = self._resize_mask(
                food["food_mask"], image_shape
            )
            total_food_mask |= mask

        if not total_food_mask.any():
            raise RuntimeError(
                "The existing SAM3 food masks are empty."
            )

        return total_food_mask

    # ---------------------------------------------------------
    # STAGE 8: DETECT INGREDIENTS INSIDE THE FOOD REGION
    # ---------------------------------------------------------
    def detect_ingredients(
        self,
        image_path,
        total_food_mask,
        image_shape,
        confidence_threshold=0.25,
    ):
        """
        Run the ingredient YOLO model and constrain each
        detection to the already-computed SAM3 food region.

        Uses per-instance YOLO segmentation masks when available.
        If the checkpoint is detection-only, bounding boxes are
        used as a rough fallback.
        """
        height, width = image_shape[:2]

        results = self.ingredient_model.predict(
            source=image_path,
            conf=confidence_threshold,
            verbose=False,
        )

        if not results:
            return []

        result = results[0]
        if result.boxes is None or len(result.boxes) == 0:
            return []

        # Segmentation masks, if this is a YOLO-seg checkpoint.
        model_masks = None
        if result.masks is not None:
            model_masks = (
                result.masks.data.detach().cpu().numpy()
            )

        names = self.ingredient_model.names
        detections = []

        for index, box in enumerate(result.boxes):
            confidence = float(box.conf[0])
            class_id = int(box.cls[0])

            if isinstance(names, dict):
                ingredient_name = str(names[class_id])
            else:
                ingredient_name = str(names[class_id])

            x1, y1, x2, y2 = map(
                int, box.xyxy[0].tolist()
            )

            if model_masks is not None and index < len(model_masks):
                # Preferred: actual per-instance ingredient mask.
                ingredient_mask = self._resize_mask(
                    model_masks[index] > 0.5,
                    image_shape,
                )
            else:
                # Fallback for a detection-only model.
                # This is approximate because the box includes
                # background pixels.
                ingredient_mask = np.zeros(
                    (height, width), dtype=bool
                )

                x1 = max(0, min(x1, width))
                x2 = max(0, min(x2, width))
                y1 = max(0, min(y1, height))
                y2 = max(0, min(y2, height))

                if x2 > x1 and y2 > y1:
                    ingredient_mask[y1:y2, x1:x2] = True

            # Only count pixels inside the existing SAM3 region.
            ingredient_mask &= total_food_mask

            if not ingredient_mask.any():
                continue

            detections.append({
                "ingredient": ingredient_name,
                "class_id": class_id,
                "confidence": confidence,
                "mask": ingredient_mask,
            })

        return detections

    # ---------------------------------------------------------
    # STAGE 9: CALCULATE INGREDIENT AREA PERCENTAGES
    # ---------------------------------------------------------
    def calculate_ingredient_percentages(
        self,
        detections,
        total_food_mask,
        scale_map=None,
    ):
        """
        Resolve overlapping ingredient masks by assigning
        shared pixels to the highest-confidence detection.

        If scale_map is provided, calculate depth-adjusted area
        percentages. Otherwise, use pixel-count percentages.
        """
        valid_food = total_food_mask.copy()

        if scale_map is not None:
            valid_food &= (
                np.isfinite(scale_map) & (scale_map > 0)
            )
            pixel_weights = np.square(scale_map)
        else:
            pixel_weights = np.ones(
                total_food_mask.shape, dtype=np.float32
            )

        total_area = float(
            pixel_weights[valid_food].sum()
        )

        if total_area <= 0:
            raise RuntimeError(
                "Total food area is zero."
            )

        # Per-pixel confidence ownership prevents double counting.
        confidence_map = np.full(
            total_food_mask.shape,
            -1.0,
            dtype=np.float32,
        )
        owner_map = np.full(
            total_food_mask.shape,
            -1,
            dtype=np.int32,
        )

        for index, detection in enumerate(detections):
            mask = detection["mask"] & valid_food
            confidence = detection["confidence"]

            update = mask & (confidence > confidence_map)
            confidence_map[update] = confidence
            owner_map[update] = index

        # Aggregate instances of the same ingredient class.
        summary = {}

        for index, detection in enumerate(detections):
            assigned = owner_map == index

            if not assigned.any():
                continue

            name = detection["ingredient"]

            area = float(pixel_weights[assigned].sum())
            pixels = int(assigned.sum())

            if name not in summary:
                summary[name] = {
                    "ingredient": name,
                    "area_pixels": 0,
                    "area_cm2": 0.0,
                    "detection_count": 0,
                    "confidence_sum": 0.0,
                }

            summary[name]["area_pixels"] += pixels
            summary[name]["area_cm2"] += area
            summary[name]["detection_count"] += 1
            summary[name]["confidence_sum"] += (
                detection["confidence"]
            )

        ingredient_results = []

        for item in summary.values():
            item["area_percentage"] = (
                item["area_cm2"] / total_area * 100.0
            )
            item["mean_confidence"] = (
                item["confidence_sum"]
                / item["detection_count"]
            )
            del item["confidence_sum"]

            ingredient_results.append(item)

        ingredient_results.sort(
            key=lambda item: item["area_percentage"],
            reverse=True,
        )

        assigned_area = sum(
            item["area_cm2"] for item in ingredient_results
        )

        return {
            "total_food_area": total_area,
            "ingredients": ingredient_results,
            "unassigned_area_percentage": max(
                0.0,
                (total_area - assigned_area) / total_area * 100.0,
            ),
        }

    # ---------------------------------------------------------
    # STAGES 7-9: REUSE STAGE 6 OUTPUTS, NO SAM3 INFERENCE
    # ---------------------------------------------------------
    def analyze_ingredient_percentages(
        self,
        image_path,
        food_results,
        image_shape,
        scale_map=None,
    ):
        print("\n[STAGE 7] Reusing existing SAM3 food masks...")

        total_food_mask = self.build_total_food_mask(
            food_results,
            image_shape,
        )

        print("\n[STAGE 8] Detecting ingredients...")

        detections = self.detect_ingredients(
            image_path=image_path,
            total_food_mask=total_food_mask,
            image_shape=image_shape,
        )

        if not detections:
            print("No ingredient detections found.")
            return {
                "total_food_mask": total_food_mask,
                "ingredient_detections": [],
                "ingredient_summary": [],
                "unassigned_area_percentage": 100.0,
            }

        print("\n[STAGE 9] Calculating ingredient percentages...")

        summary = self.calculate_ingredient_percentages(
            detections=detections,
            total_food_mask=total_food_mask,
            scale_map=scale_map,
        )

        for item in summary["ingredients"]:
            print(
                f"  {item['ingredient']}: "
                f"{item['area_percentage']:.2f}% "
                f"(confidence={item['mean_confidence']:.3f})"
            )

        print(
            "  Unassigned food area: "
            f"{summary['unassigned_area_percentage']:.2f}%"
        )

        return {
            "total_food_mask": total_food_mask,
            "ingredient_detections": detections,
            "ingredient_summary": summary["ingredients"],
            "total_food_area": summary["total_food_area"],
            "unassigned_area_percentage": (
                summary["unassigned_area_percentage"]
            ),
        }
    def calculate_food_volume(self, base_area_cm2):
        """
        Estimate food volume by approximating the food as a cone.

        Args:
            base_area_cm2: Estimated base area in cm².

        Returns:
            Estimated volume in cm³.
        """
        volume_cm3 = (
            base_area_cm2 * self.CONE_HEIGHT_CM / 3.0
        )

        return float(volume_cm3)
    # ---------------------------------------------------------
    # RUN ALL SIX STAGES
    # ---------------------------------------------------------
    def estimate(self, image_path):
        image = cv2.imread(image_path)

        if image is None:
            raise FileNotFoundError(
                f"Could not read image: {image_path}"
            )

        image_shape = image.shape

        # Stage 1
        foods = self.detect_food(image_path)

        # Stage 2
        utensil = self.detect_utensil(image_path)

        # Stage 3
        utensil_data = self.segment_utensil(
            image_path,
            utensil,
        )

        # Stage 4
        depth_map = self.get_depth_map(
            image_path,
            image_shape,
        )

        # Stage 5
        calibration = self.calculate_scale_map(
            depth_map=depth_map,
            utensil_mask=utensil_data["mask"],
            utensil_length_px=utensil_data["length_px"],
        )

        scale_map = calibration["scale_map"]

        # Stage 6
        food_results = []

        print("\n[STAGE 6] Calculating food areas...")

        for index, food in enumerate(foods):
            try:
                food_mask = self.segment_food_item(
                    image_path,
                    food,
                    image_shape,
                )

                measurements = self.calculate_food_area(
                    food_mask,
                    scale_map,
                )

                item = {
                    "food_type": food["name"],
                    "confidence": food["confidence"],
                    "bbox": (
                        food["x1"],
                        food["y1"],
                        food["x2"],
                        food["y2"],
                    ),
                    "food_mask": measurements["food_mask"],
                    "food_area_pixels": (
                        measurements["area_pixels"]
                    ),
                    "food_area_cm2": (
                        measurements["area_cm2"]
                    ),
                }

                food_results.append(item)

                print(
                    f"  {index + 1}. {food['name']}: "
                    f"{item['food_area_cm2']:.2f} cm² "
                    f"({item['food_area_pixels']} valid pixels)"
                )

            except RuntimeError as exc:
                print(
                    f"  Could not estimate {food['name']}: {exc}"
                )

        if not food_results:
            raise RuntimeError(
                "No food areas could be calculated."
            )
        # Stages 7-9:
        # Reuse food masks already generated in Stage 6.
        # Do not run SAM3 again.
        ingredient_analysis = self.analyze_ingredient_percentages(
            image_path=image_path,
            food_results=food_results,
            image_shape=image_shape,
            scale_map=scale_map,
        )
        for item in food_results:
            item["volume_cm3"] = self.calculate_food_volume(
                base_area_cm2=item["food_area_cm2"]
            )
        return {
            "foods": food_results,
            "utensil": utensil,
            "utensil_mask": utensil_data["mask"],
            "utensil_length_px": utensil_data["length_px"],
            "assumed_utensil_length_cm": (
                self.UTENSIL_LENGTH_CM
            ),
            "depth_map": depth_map,
            "scale_map_cm_per_pixel": scale_map,
            "reference_depth": calibration["reference_depth"],
            "reference_cm_per_pixel": (
                calibration["reference_cm_per_pixel"]
            ),
            "ingredient_analysis": ingredient_analysis,
            "ingredient_summary": ingredient_analysis["ingredient_summary"],
            "total_food_mask": ingredient_analysis["total_food_mask"],
            
        }


# =============================================================
# RUN
# =============================================================


FOOD_MODEL_PATH = r"C:\Study\Major\Food\classif.pt"

UTENSIL_MODEL_PATH = (
        r"C:\Study\Major\runs\detect\runs"
        r"\spoon_yolo11m\weights\last.pt"
    )

SAM_MODEL_PATH = r"C:\Study\Major\sam3.pt"

IMAGE_PATH = r"C:\Study\Major\Spoon\Tests\Test3.jpeg"
pipeline = FoodAreaPipeline(
        food_model_path=FOOD_MODEL_PATH,
        utensil_model_path=UTENSIL_MODEL_PATH,
        sam_model_path=SAM_MODEL_PATH,
        depth_api_url="http://localhost:8000/predict-depth-map",
        ingredient_model_path=(
            r"C:\Study\Major\best (1).pt"
        ),
    )

output = pipeline.estimate(IMAGE_PATH)

print("\n========== FINAL RESULTS ==========")

for item in output["foods"]:
        print(
            f"{item['food_type']}: "
            f"{item['food_area_cm2']:.2f} cm²"
        )
print("\n========== INGREDIENT AREA BREAKDOWN ==========")

for item in output["ingredient_summary"]:
        print(
            f"{item['ingredient']}: "
            f"{item['area_percentage']:.2f}%"
        )

print(
        "Unassigned area: "
        f"{output['ingredient_analysis']['unassigned_area_percentage']:.2f}%"
    )


Initializing food area pipeline...

Loading YOLO model...
YOLO model loaded successfully

Loading SAM 3...
SAM 3 loaded successfully

Loading SAM 3...
Pipeline initialized.

[STAGE 1] Detecting food types...

image 1/1 C:\Study\Major\Spoon\Tests\Test3.jpeg: 512x480 1 poha, 84.6ms
Speed: 3.6ms preprocess, 84.6ms inference, 3.1ms postprocess per image at shape (1, 3, 512, 480)
  Food 1: poha (confidence=0.949)

[STAGE 2] Detecting spoon/fork...

image 1/1 C:\Study\Major\Spoon\Tests\Test3.jpeg: 640x608 1 Spoon, 23.0ms
Speed: 6.2ms preprocess, 23.0ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 608)
  Utensil: spoon (confidence=0.948)

[STAGE 3] Segmenting utensil with SAM 3...
Ultralytics 8.4.118  Python-3.14.6 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Study\Major\Spoon\Tests\Test3.jpeg: 644x644 1 spoon, 590.2ms
Speed: 8.9ms preprocess, 590.2ms inference,

In [12]:
print(output["foods"])

[{'food_type': 'poha', 'confidence': 0.9494404196739197, 'bbox': (62.79640197753906, 451.69561767578125, 1019.7351684570312, 1359.32275390625), 'food_mask': array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]], shape=(1600, 1479)), 'food_area_pixels': 543121, 'food_area_cm2': 79.27302551269531, 'volume_cm3': 132.12170918782553}]


In [15]:
vol = output["foods"][0]["volume_cm3"]
food_type = output["foods"][0]["food_type"]

In [ ]:
vol


132.12170918782553

In [16]:
food_type

'poha'

In [22]:
import json 

allergies_data = json.load(open(r"C:\Study\Major\Working\Metrics\Allergies.json", "r"))
density_data = json.load(open(r"C:\Study\Major\Working\Metrics\Density.json", "r"))
food_types_data = json.load(open(r"C:\Study\Major\Working\Metrics\Types.json", "r"))


In [29]:
def get_density(food_type):
    """
    Retrieve the density of a food type from the density data.

    Args:
        food_type (str): The name of the food type.

    Returns:
        float: The density in g/cm³, or None if not found.
    """
    return density_data.get(food_type.lower()).get("density_class")
print(f"Density of {food_type}: {get_density(food_type)} g/cm³")

Density of poha: Low g/cm³


In [30]:
Density_classes = json.loads(open(r"C:\Study\Major\Working\Metrics\DensityRange.json", "r").read())

In [31]:
dclass = get_density(food_type)


In [32]:
def match_density(density_class):
    return Density_classes[density_class]["lower_bound"], Density_classes[density_class]["upper_bound"]

density = match_density(dclass)

In [37]:
quantity = vol*((density[0]+density[1])/2)
print(quantity)



59.454769134521484


In [39]:

import json
from pathlib import Path


DATABASE_PATH = Path(r"C:\Study\Major\Working\Metrics\Ingredients.json")

with DATABASE_PATH.open("r", encoding="utf-8") as file:
    FOOD_DB = json.load(file)

# Ingredient nutrient profiles, each expressed per 100 g.
INGREDIENT_PROFILES = FOOD_DB["_ingredient_profiles_per_100g"]

NUTRIENTS = FOOD_DB["_metadata"]["nutrients"]


def calculate_dish_nutrients(dish_name: str, target_weight_g: float = 100.0):
    """
    Calculate ingredient quantities and total nutrients for a dish.

    Args:
        dish_name: Classifier output, e.g. 'palak_paneer'.
        target_weight_g: Desired prepared dish weight in grams.

    Returns:
        Dictionary containing ingredient quantities, nutrient contributions,
        and total nutrients for the requested dish weight.
    """

    if target_weight_g <= 0:
        raise ValueError("target_weight_g must be greater than zero.")

    dish_name = dish_name.strip().lower()

    if dish_name not in FOOD_DB or dish_name.startswith("_"):
        raise ValueError(f"Unknown dish: {dish_name}")

    dish = FOOD_DB[dish_name]
    recipe = dish["ingredients"]

    if not recipe:
        raise ValueError(f"No ingredients defined for {dish_name}")

    # Validate and normalize recipe proportions.
    proportions = {}

    for ingredient, fraction in recipe.items():
        if ingredient not in INGREDIENT_PROFILES:
            raise ValueError(
                f"Missing nutrient profile for ingredient: {ingredient}"
            )

        if not isinstance(fraction, (int, float)) or fraction < 0:
            raise ValueError(
                f"Invalid proportion for {ingredient}: {fraction}"
            )

        proportions[ingredient] = float(fraction)

    proportion_sum = sum(proportions.values())

    if proportion_sum <= 0:
        raise ValueError("Recipe proportions must sum to a positive value.")

    proportions = {
        ingredient: fraction / proportion_sum
        for ingredient, fraction in proportions.items()
    }

    # Scale ingredient quantities to the requested total weight.
    ingredient_quantities = {
        ingredient: fraction * target_weight_g
        for ingredient, fraction in proportions.items()
    }

    # Calculate nutrient contributions from each ingredient.
    ingredient_breakdown = {}
    total_nutrients = {nutrient: 0.0 for nutrient in NUTRIENTS}

    for ingredient, quantity_g in ingredient_quantities.items():
        profile = INGREDIENT_PROFILES[ingredient]

        contribution = {}

        for nutrient in NUTRIENTS:
            # Nutrient profile is per 100 g of ingredient.
            value_per_100g = profile[nutrient]
            amount = value_per_100g * quantity_g / 100.0

            contribution[nutrient] = amount
            total_nutrients[nutrient] += amount

        ingredient_breakdown[ingredient] = {
            "quantity_g": round(quantity_g, 4),
            "nutrients": {
                nutrient: round(value, 4)
                for nutrient, value in contribution.items()
            }
        }

    # Ensure ingredient quantities balance to the requested weight.
    total_ingredient_weight = sum(ingredient_quantities.values())

    # The total nutrient amounts are for target_weight_g grams of dish.
    # Convert them to a per-100-g basis if the requested weight differs.
    nutrients_per_100g = {
        nutrient: value * 100.0 / target_weight_g
        for nutrient, value in total_nutrients.items()
    }

    return {
        "dish": dish_name,
        "base_type": dish["base_type"],
        "target_weight_g": target_weight_g,
        "total_ingredient_weight_g": round(total_ingredient_weight, 4),
        "ingredients": ingredient_breakdown,
        "total_nutrients": {
            nutrient: round(value, 4)
            for nutrient, value in total_nutrients.items()
        },
        "nutrients_per_100g": {
            nutrient: round(value, 4)
            for nutrient, value in nutrients_per_100g.items()
        }
    }


if __name__ == "__main__":
    result = calculate_dish_nutrients("palak_paneer", 100)

    print(json.dumps(result, indent=2))


{
  "dish": "palak_paneer",
  "base_type": "gravy",
  "target_weight_g": 100,
  "total_ingredient_weight_g": 100.0,
  "ingredients": {
    "spinach": {
      "quantity_g": 50.0,
      "nutrients": {
        "carbohydrates_g": 1.8,
        "fat_g": 0.2,
        "protein_g": 1.45,
        "fibre_g": 1.1,
        "sugar_g": 0.2
      }
    },
    "paneer": {
      "quantity_g": 25.0,
      "nutrients": {
        "carbohydrates_g": 0.5625,
        "fat_g": 6.125,
        "protein_g": 4.375,
        "fibre_g": 0.0,
        "sugar_g": 0.5625
      }
    },
    "curry_gravy": {
      "quantity_g": 15.0,
      "nutrients": {
        "carbohydrates_g": 0.78,
        "fat_g": 1.71,
        "protein_g": 0.285,
        "fibre_g": 0.105,
        "sugar_g": 0.525
      }
    },
    "onion": {
      "quantity_g": 5.0,
      "nutrients": {
        "carbohydrates_g": 0.465,
        "fat_g": 0.005,
        "protein_g": 0.055,
        "fibre_g": 0.085,
        "sugar_g": 0.21
      }
    },
    "oil": {
